# PIMMS — phase separation in your browser 🧪

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/holehouse-lab/PIMMS/blob/master/colab/pimms_phase_separation_demo.ipynb)

This notebook runs a small **[PIMMS](https://github.com/holehouse-lab/PIMMS)** lattice
Monte Carlo simulation of a system of "sticky" polymers that **phase separate** into a
dense droplet coexisting with a dilute phase — the lattice analogue of a biomolecular
condensate — and then lets you **view the trajectory in 3D**, right here in the Colab
session.

The whole thing takes about a minute end to end (most of that is compiling PIMMS'
Cython kernels on first install).

**What you'll do**
1. Install PIMMS from PyPI
2. Define a simple sticky-polymer system (a keyfile + a parameter file)
3. Run the simulation
4. Watch the droplet in an interactive 3D viewer
5. Quantify the phase separation with the bundled `lemonade` analysis package

## 1. Install PIMMS

`idptools-pimms` is on PyPI. Installing it compiles PIMMS' native Cython kernels, so the
first install takes ~1 minute. We also grab `py3Dmol` for the interactive 3D viewer
(`mdtraj`, `numpy` and `scipy` come in automatically as PIMMS dependencies).

In [ ]:
!pip install -q idptools-pimms py3Dmol

## 2. Define the system

A PIMMS run is driven by two plain-text files:

* a **parameter file** (`params.prm`) — the "force field". Here there is a single bead
  type `A` with a favourable `A–A` contact energy (so chains like to stick to each other)
  and a neutral interaction with solvent (type `0`).
* a **keyfile** (`KEYFILE.kf`) — the simulation setup: a 22×22×22 periodic box holding
  120 eight-bead chains (~9% occupancy), sampled with a mix of local **crankshaft** moves
  and whole-chain **reptation** ("slither") moves.

The contact energy is strong enough, relative to the temperature, that the chains demix
from the solvent and collapse into a single dense droplet.

In [ ]:
%%writefile params.prm
# A single "sticky" bead type: A likes A, and is neutral toward solvent (type 0).
A  A  -8      # favourable A-A short-range contact energy
A  0   0      # A-solvent (solvation) - required for every bead type

In [ ]:
%%writefile KEYFILE.kf
DIMENSIONS      : 22 22 22
PARAMETER_FILE  : params.prm
CHAIN           : 120 AAAAAAAA      # 120 copies of an 8-bead sticky homopolymer
TEMPERATURE     : 30
ANGLES_OFF      : True              # no backbone-angle penalties in this demo

N_STEPS         : 250
EQUILIBRATION   : 20

# moves (the MOVE_* fractions must sum to 1.0)
MOVE_CRANKSHAFT     : 0.6           # local bead moves (the workhorse)
CRANKSHAFT_SUBSTEPS : 15000
MOVE_SLITHER        : 0.4           # whole-chain reptation, for faster mixing
SLITHER_SUBSTEPS    : 30

# output / analysis frequencies (in steps)
EN_FREQ         : 5
XTC_FREQ        : 5
ANA_CLUSTER     : 5
SEED            : 5

## 3. Run the simulation

This writes an `ENERGY.dat` trace, a `START.pdb` + `traj.xtc` trajectory, and a set of
analysis files into the working directory. It should take only a few seconds.

In [ ]:
!PIMMS -k KEYFILE.kf

## 4. Watch the droplet (interactive 3D)

We load the trajectory with `mdtraj`, convert it to a multi-model PDB, and animate it with
`py3Dmol`. Each sphere is a bead and colours distinguish chains. **Drag to rotate, scroll
to zoom**, and use the play controls to step through the trajectory and watch the sticky
chains gather into a single droplet.

In [ ]:
import mdtraj as md
import py3Dmol

traj = md.load("traj.xtc", top="START.pdb")
traj.save_pdb("traj_movie.pdb")            # multi-model PDB: one MODEL per frame
movie = open("traj_movie.pdb").read()

view = py3Dmol.view(width=760, height=520)
view.addModelsAsFrames(movie)
view.setStyle({'sphere': {'radius': 0.5, 'colorscheme': 'chainHetatm'}})
view.setBackgroundColor('0xffffff')
view.zoomTo()
view.animate({'loop': 'forward', 'interval': 120})
view.show()

## 5. Quantify the phase separation

PIMMS ships with **`lemonade`**, a fast trajectory-analysis package. Loading the run
(from the same xtc / pdb / keyfile) gives the standard order parameters for phase
separation:

* the **energy** falling as contacts form,
* the **condensed fraction** — the fraction of beads in the largest cluster,
* the **radial density profile** — occupied-site fraction vs distance from the droplet
  centre, which falls from a dense core to a dilute background: the signature of a
  coexisting dense + dilute phase.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pimms.lemonade as lemonade
from pimms.lemonade import phase_separation as ps

lt = lemonade.load(xtc="traj.xtc", pdb="START.pdb", keyfile="KEYFILE.kf", verbose=False)

energy = np.loadtxt("ENERGY.dat")                  # columns: step, energy
cf = ps.condensed_fraction(lt)                     # per frame
second_half = lt[lt.n_frames // 2:]                # measure the equilibrated droplet
r, rho = ps.radial_density_profile(second_half)
result = ps.analyze(second_half)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(energy[:, 0], energy[:, 1], color="#921e2c")
ax[0].set(xlabel="MC step", ylabel="potential energy", title="Energy — contacts forming")
ax[1].plot(cf, marker="o", ms=3, color="#921e2c")
ax[1].set(xlabel="frame", ylabel="condensed fraction",
          title="Beads in the largest cluster", ylim=(0, 1))
ax[2].plot(r, rho, marker="o", color="#921e2c")
ax[2].set(xlabel="distance from droplet centre (lattice units)",
          ylabel="occupied fraction", title="Radial density profile", ylim=(0, 1))
plt.tight_layout()
plt.show()

print(f"geometry             : {result.geometry}")
print(f"dense-phase density  : {result.rho_dense:.2f}")
print(f"dilute-phase density : {result.rho_dilute:.2f}")
print(f"phase separated?     : {result.is_phase_separated}")

## Where to go next

* **Full documentation:** https://idptools-pimms.readthedocs.io
* **Experiment** — re-run with different knobs: lower the `TEMPERATURE` (or make the `A A`
  energy more negative) for a tighter droplet; raise the temperature until the droplet
  dissolves; or add a second bead type with its own interactions to explore
  multi-component and multiphase behaviour.
* `PIMMS --info` lists every keyfile keyword; the
  [Input files](https://idptools-pimms.readthedocs.io/en/latest/input_files.html) and
  [keyword reference](https://idptools-pimms.readthedocs.io/en/latest/keywords.html)
  pages document them in full.
* `lemonade` does much more than shown here — single-chain conformational analysis,
  surface tension, coexistence (binodal) densities, and more.